# Chain-of-Thought Prompting

A refresher on getting LLMs to **reason in intermediate steps** before answering — why it
works, the variants that matter (zero-shot CoT, self-consistency), how to parse the answer
out, and when it's the wrong tool.

**Domain:** LLM Inference, Training & Optimization  ·  **runnable:** yes

## 1. What & Why

**What.** Chain-of-Thought (CoT) prompting asks a model to produce intermediate reasoning
steps — "show your work" — *before* committing to a final answer, instead of emitting the
answer directly. You either supply few-shot exemplars that contain worked reasoning, or, in
the zero-shot variant, simply append a trigger like *"Let's think step by step."*

**The problem it solves.** A single forward pass gives the model a fixed amount of
computation per token. For a multi-step problem (arithmetic, logic, planning, multi-hop QA),
forcing a one-shot answer makes the model commit before it has "spent" enough tokens to work
the problem out. Emitting reasoning tokens turns the autoregressive loop into a scratchpad:
each step conditions the next, so later tokens can attend to partial results. Wei et al.
(2022) showed this unlocks large accuracy jumps on GSM8K / arithmetic / commonsense
benchmarks — but only for sufficiently large models; small models often produce fluent-but-
wrong chains.

**When to reach for it.** Multi-step math, symbolic/logical reasoning, multi-hop retrieval
synthesis, anything where a human would need scratch paper. **When not to:** lookups,
classification, formatting, extraction — tasks with no intermediate structure. There CoT
just burns tokens, adds latency, and gives the model more room to talk itself into an error.

## 2. Mental Model

Think of the context window as a **scratchpad the model writes on, then reads back**.

```
Direct:    [question] -> [answer]           # one leap, fixed compute
CoT:       [question] -> [step][step]... -> [answer]
                          └── each step is read back by the next ──┘
```

A Transformer does a *constant* amount of work per generated token. The only way to spend
**more** compute on a hard problem is to **generate more tokens**. Reasoning steps are that
extra compute, externalized into the sequence where attention can reuse them. The final
answer is then conditioned on a sequence that already contains the sub-results — it's
copying/combining, not deriving from scratch.

Corollary: the answer is now buried at the *end* of a long generation, so you always need a
**parser** (a regex, a delimiter convention like `#### 42`, or a structured-output schema)
to pull it back out.

## 3. Key Concepts

- **Zero-shot CoT** — append a trigger ("Let's think step by step") with no exemplars
  (Kojima et al. 2022). Cheapest to deploy; works surprisingly well on large models.
- **Few-shot CoT** — provide a handful of `question -> reasoning -> answer` exemplars so the
  model imitates the *format and granularity* of the reasoning (Wei et al. 2022).
- **Self-consistency** — sample *N* reasoning chains at temperature > 0, then **majority-vote
  the final answers** (Wang et al. 2022). Different chains, same correct destination; outvotes
  the occasional bad chain. The single biggest accuracy lever on top of plain CoT.
- **Answer delimiter** — a fixed marker (e.g. `#### <answer>` or "The answer is X") so the
  final answer is mechanically extractable from a long trace.
- **Reasoning vs. answer tokens** — only the final answer is "graded"; the reasoning is
  scaffolding. With reasoning models (o-series, Claude extended thinking) the scaffold is
  produced and billed as separate *thinking* tokens.
- **Emergence with scale** — CoT helps large models and can *hurt* small ones, which
  generate confident but invalid chains. It's not free for every model size.
- **Faithfulness caveat** — the printed chain is a *plausible* rationalization, not a
  guaranteed trace of the computation. Don't treat it as a proof or an audit log.

## 4. Setup

The local examples are **stdlib only** — they simulate model behavior deterministically on
CPU so the notebook runs anywhere. The optional live cell uses the Anthropic SDK and is
gated behind an `ANTHROPIC_API_KEY` check, so the notebook executes top-to-bottom either way.

```bash
# only needed if you want to run the live Claude cell at the end
%pip install anthropic
```

In [ ]:
import os
import re
import textwrap
from collections import Counter

print("stdlib only — no install needed for the local examples")
print("ANTHROPIC_API_KEY set:", bool(os.getenv("ANTHROPIC_API_KEY")))

## 5. Worked Examples

### Example 1 — Why steps beat a one-shot guess

We use a **toy word problem** and a deterministic stand-in "model". The point isn't the
arithmetic (that's trivial in Python) — it's the *shape*: a CoT response emits intermediate
results, and we must **parse the final answer out** of the trace with a delimiter convention.
Compare a direct answer (just a number, easy to get wrong if mis-derived) against a chain
that lays out each step and ends with a `#### <answer>` marker.

In [ ]:
PROBLEM = (
    "A cafe had 23 muffins. They sold 17 in the morning, baked 30 more, "
    "then sold 12 in the afternoon. How many muffins are left?"
)

# A direct-answer prompt asks for the number only.
direct_prompt = PROBLEM + "\nAnswer with just the number."

# A CoT prompt asks for steps, then a delimited final answer.
cot_prompt = PROBLEM + textwrap.dedent("""
    Let's think step by step. Show each calculation on its own line,
    then give the final answer on a new line as: #### <number>""")


def stand_in_cot_model(prompt):
    """Toy deterministic 'model'. If asked to reason, it externalizes each step
    onto the scratchpad and ends with the #### delimiter; otherwise it guesses."""
    if "step by step" in prompt:
        return textwrap.dedent("""\
            Start: 23 muffins.
            Sold 17 in the morning: 23 - 17 = 6.
            Baked 30 more: 6 + 30 = 36.
            Sold 12 in the afternoon: 36 - 12 = 24.
            #### 24""")
    # Direct mode: imagine the model fixates on the last two numbers and slips.
    return "18"


def parse_answer(text):
    """Pull the final answer: prefer the #### delimiter, else the last integer."""
    m = re.search(r"####\s*(-?\d+)", text)
    if m:
        return int(m.group(1))
    nums = re.findall(r"-?\d+", text)
    return int(nums[-1]) if nums else None


direct_out = stand_in_cot_model(direct_prompt)
cot_out = stand_in_cot_model(cot_prompt)

print("DIRECT answer:", parse_answer(direct_out))
print("\nCoT trace:\n" + cot_out)
print("\nCoT parsed answer:", parse_answer(cot_out))

assert parse_answer(cot_out) == 24  # 23 - 17 + 30 - 12

### Example 2 — Self-consistency: sample many chains, majority-vote

Real CoT chains sampled at temperature > 0 are **noisy**: most land on the right answer, a
few go astray. Self-consistency exploits this — instead of trusting one chain, sample several
and take the **majority vote** over their final answers. Below we simulate five sampled
chains for the same problem (four correct, one arithmetic slip) and show that voting recovers
the right answer even though any single sample might be wrong.

In [ ]:
# Five independently "sampled" reasoning traces for the muffin problem.
# Four reason correctly to 24; one slips (forgets to subtract the afternoon sales).
sampled_chains = [
    "23 - 17 = 6; 6 + 30 = 36; 36 - 12 = 24. #### 24",
    "23 - 17 = 6; 6 + 30 = 36; 36 - 12 = 24. #### 24",
    "23 - 17 = 6; 6 + 30 = 36. Forgot the afternoon. #### 36",   # bad chain
    "Morning leaves 6; +30 = 36; -12 = 24. #### 24",
    "23 minus 17 is 6, plus 30 is 36, minus 12 is 24. #### 24",
]

answers = [parse_answer(c) for c in sampled_chains]
votes = Counter(answers)
final, count = votes.most_common(1)[0]

print("per-chain answers:", answers)
print("vote tally:", dict(votes))
print(f"self-consistency answer: {final}  ({count}/{len(answers)} chains agree)")

# A single greedy sample could have been the bad chain (36); voting fixes that.
assert final == 24

### Example 3 — Zero-shot CoT against a live model (gated)

The same idea with a real model: a single user turn plus the *"Let's think step by step"*
trigger, then parse the delimited answer from the response. This cell only fires when
`ANTHROPIC_API_KEY` is set; otherwise it prints the call shape so the notebook still runs
cleanly.

In [ ]:
def solve_with_cot(question):
    import anthropic  # imported lazily so the notebook runs without the package
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
    resp = client.messages.create(
        model="claude-opus-4-8",
        max_tokens=512,  # leave room for the reasoning trace, not just the answer
        messages=[{
            "role": "user",
            "content": question + "\n\nLet's think step by step, then end with: "
                                   "#### <number>",
        }],
    )
    return resp.content[0].text


if os.getenv("ANTHROPIC_API_KEY"):
    trace = solve_with_cot(PROBLEM)
    print(trace)
    print("\nparsed answer:", parse_answer(trace))
else:
    print("ANTHROPIC_API_KEY not set — skipping the live call.")
    print("Would send model=claude-opus-4-8 with the question +")
    print('"Let\'s think step by step, then end with: #### <number>"')
    print("then parse the final answer from the returned trace with parse_answer().")

## 6. Gotchas & Pitfalls

- **Forgetting to parse.** The answer is now at the *end* of a long generation. Without a
  delimiter (`####`, "The answer is …") or structured output, you'll regex the wrong number.
  Always pin the format and extract deterministically.
- **CoT on the wrong task.** For classification/extraction/lookups it adds latency, cost, and
  *more* surface area to hallucinate — often *lowering* accuracy. Reserve it for genuinely
  multi-step problems.
- **`max_tokens` too small.** Reasoning needs room. Cap the budget for a direct answer and the
  model gets truncated mid-chain, emitting no final answer at all.
- **Greedy decoding + one sample.** Plain CoT at temperature 0 still fails on the hard tail.
  If accuracy matters, add **self-consistency** (sample N, majority-vote) before reaching for
  a bigger model.
- **Treating the chain as truth.** Printed reasoning is a plausible story, not a faithful log
  of the internal computation — models reach right answers via wrong-looking chains and vice
  versa. Don't surface it to users as a guarantee or use it as an audit trail.
- **Small-model CoT.** Below a capability threshold, CoT produces confident invalid chains and
  can hurt. Verify the lift on *your* model rather than assuming it.
- **Leaking the scratchpad.** Users rarely want the full chain. Strip everything before the
  delimiter (or use a model's separate *thinking* channel) before showing the answer.

## 7. When to Use vs Alternatives

| Approach | Best for | Cost / trade-off |
|---|---|---|
| **Direct prompt** | lookups, classification, extraction, formatting | cheapest & fastest; fails on multi-step reasoning |
| **Zero-shot CoT** (`"think step by step"`) | quick reasoning lift, no exemplars handy | a few extra tokens; format of reasoning is uncontrolled |
| **Few-shot CoT** | need a specific reasoning style/granularity | exemplars eat context; must curate good ones |
| **Self-consistency** | squeeze max accuracy on reasoning tasks | N× the cost/latency (N samples); needs a parseable answer |
| **Reasoning models** (o-series, Claude extended thinking) | hardest reasoning, don't want to hand-craft prompts | thinking tokens are billed; less prompt control, more latency |
| **Tool use / Program-aided** (PAL, code execution) | exact arithmetic/logic, verifiable steps | offloads computation to a tool; needs a runtime + plumbing |

Rules of thumb: start **direct**; if the task is multi-step, add **zero-shot CoT**; if you
need the last few points of accuracy, layer **self-consistency**; if the model can *call
code/tools*, prefer that for anything requiring exact arithmetic — a Python eval beats any
amount of token-by-token mental math. Modern **reasoning models** internalize much of CoT, so
on those the explicit "think step by step" prompt is often redundant (and you pay for the
thinking either way).

## 8. Resources

- **Chain-of-Thought Prompting Elicits Reasoning in LLMs** — Wei et al., 2022 (the original
  paper): https://arxiv.org/abs/2201.11903
- **Self-Consistency Improves Chain of Thought Reasoning** — Wang et al., 2022:
  https://arxiv.org/abs/2203.11171
- **Large Language Models are Zero-Shot Reasoners** — Kojima et al., 2022 (the
  "Let's think step by step" trigger): https://arxiv.org/abs/2205.11916
- **Prompt Engineering Guide — Chain-of-Thought** (practical walkthrough):
  https://www.promptingguide.ai/techniques/cot
- **Anthropic docs — Let Claude think (chain of thought) / extended thinking:**
  https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/chain-of-thought
- Related notebooks in this domain: `prompt-engineering.ipynb` (prompt structure, parsing)
  and `few-shot-learning.ipynb` (exemplar selection).